# Wholesale Electricity Market: Multiplier Analysis and Dynamic Simulation with PROC SIMLIN

## Executive Summary

A utility planner models a regional wholesale power market as a three-equation simultaneous system (demand, wholesale price, renewable generation) with lagged feedback. Structural coefficients are estimated by two-stage least squares in PROC SYSLIN, then PROC SIMLIN derives the reduced form, computes impact/interim/total multipliers for exogenous policy drivers, and runs a dynamic simulation to reproduce the market's historical path.

## Data Sources

| Dataset | Rows | Description |
|---------|------|-------------|
| `power_market` | 60 | Monthly synthetic wholesale-market panel (2021-2025) for one balancing area. Three jointly determined (endogenous) series — system `demand` (MW), `price` (wholesale $/MWh), `renewgen` (renewable output, MW) — driven by four exogenous variables: `cdd` (cooling degree days), `gasprice` ($/MMBtu), `windindex` (normalized wind availability), and `capacity` (installed renewable nameplate). Generated with `call streaminit(20260531)` and `rand('normal')` shocks; demand and price feed back on their own prior-month values to create genuine lag dynamics. |
| `market_lags` | 60 | `power_market` augmented with one-period lags `p_lag1` and `d_lag1` (built with `lag()`, seeded in month 1), used both as instruments/regressors in SYSLIN and as the lagged endogenous terms in SIMLIN. |
| `struct` | 3 | TYPE=EST structural-coefficient dataset written by `PROC SYSLIN ... OUTEST=` and consumed by SIMLIN via `EST=`. |
| `redform` | — | SIMLIN `OUTEST=` output holding reduced-form coefficients and impact/interim/total multipliers. |
| `sim_out` | 60 | SIMLIN `OUTPUT OUT=` dataset of predicted (`*_hat`) and residual (`*_res`) values for each endogenous variable. |

# Wholesale Electricity Market: Multiplier Analysis and Dynamic Simulation

Regional power markets are inherently **simultaneous**: load and wholesale price are determined together (price clears the market, but price also shapes demand), and renewable output responds to price while displacing the thermal generation that sets price. Estimating any single equation in isolation produces biased coefficients because of this feedback.

In this notebook we:

1. Generate a synthetic 5-year monthly panel for one balancing area.
2. Estimate a three-equation structural system by **two-stage least squares** (`PROC SYSLIN`).
3. Feed the structural estimates to **`PROC SIMLIN`** to obtain the **reduced form**, the **impact / interim / total multipliers** of each exogenous driver, and a **dynamic simulation** of the market.

The multipliers answer the planner's real question: *if cooling demand or gas prices move, how much do load, price, and renewable output ultimately change — this month, and after the lag dynamics have fully played out?*

## Step 0 — Generate the synthetic market panel

We build 60 monthly observations. Three structural relationships govern the system:

- **Demand** falls with the prior-month price (consumers adjust slowly), rises with cooling degree days (`cdd`), and is persistent (depends on its own lag).
- **Renewable generation** rises with wind availability, installed capacity, and is mildly price-responsive (more dispatch when prices are high).
- **Price** clears the market: it rises with demand and the marginal gas price, and falls as cheap renewable output enters the stack.

Normally distributed shocks (`rand('normal')`) make each equation stochastic. We carry `pricelag` and `demandlag` forward each iteration so the series have authentic month-to-month dynamics.

In [1]:
/* Synthetic regional wholesale electricity market: 5 years, monthly */
data power_market;
   call streaminit(20260531);

   /* Pre-sample seeds for the lagged feedback terms */
   pricelag  = 48.0;    /* lagged wholesale price ($/MWh) */
   demandlag = 1050.0;  /* lagged system demand (MW)      */

   do month = 1 to 60;
      year = 2021 + int((month - 1) / 12);

      /* Exogenous drivers */
      cdd       = max(0, 250 + 120*sin((month-6)/12*2*3.14159)
                        + 30*rand('normal'));   /* cooling degree days */
      gasprice  = 3.2 + 0.015*month + 0.5*rand('normal'); /* $/MMBtu     */
      windindex = 0.55 + 0.12*rand('normal');             /* 0-1 scale   */
      capacity  = 1400 + 4*month;                         /* MW nameplate*/

      /* Structural shocks */
      shock_d = 18*rand('normal');
      shock_p =  4*rand('normal');
      shock_r = 25*rand('normal');

      /* Simultaneous structural system with lag feedback */
      demand   = 600 - 2.1*pricelag + 0.9*cdd + 0.40*demandlag + shock_d;
      renewgen = -120 + 1.3*pricelag + 480*windindex + 0.18*capacity + shock_r;
      price    = 12 + 0.022*demand + 7.5*gasprice - 0.030*renewgen + shock_p;

      output;

      /* Roll forward for next month's dynamics */
      pricelag  = price;
      demandlag = demand;
   end;

   keep month year demand price renewgen cdd gasprice windindex capacity;
run;

NOTE: DATA power_market


NOTE: Wrote power_market (60 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 1 — Build explicit lag columns

PROC SIMLIN's `LAGGED` statement needs the lagged endogenous values to appear as their own variables in the simulation dataset, and SYSLIN uses those same lags as predetermined regressors and instruments. We construct `p_lag1` and `d_lag1` with the `lag()` function, seeding month 1 with the same pre-sample values used during data generation.

In [2]:
data market_lags;
   set power_market;
   p_lag1 = lag(price);
   d_lag1 = lag(demand);
   if month = 1 then do;
      p_lag1 = 48.0;
      d_lag1 = 1050.0;
   end;
run;

NOTE: DATA market_lags


NOTE: Read 60 rows from power_market.
NOTE: Wrote market_lags (60 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Estimate the structural system by 2SLS

Because `demand`, `price`, and `renewgen` are jointly determined, OLS on any single equation would be biased. Two-stage least squares uses **instruments** — the exogenous drivers plus the predetermined lags — to purge the endogenous regressors of their correlation with the structural error.

The `ENDOGENOUS` statement names the jointly dependent variables, `INSTRUMENTS` lists the predetermined variables, and each `MODEL` defines one structural equation. `OUTEST=struct` writes the coefficients in the `TYPE=EST` form that PROC SIMLIN expects.

In [3]:
title 'Step 2: Structural estimation via two-stage least squares';
proc syslin 2sls data=market_lags outest=struct;
   endogenous  demand price renewgen;
   instruments cdd gasprice windindex capacity p_lag1 d_lag1;

   demandeq: model demand   = price cdd d_lag1;
   priceeq:  model price    = demand gasprice renewgen;
   reneweq:  model renewgen = price windindex capacity;
run;

proc print data=struct;
   title2 'Structural coefficient dataset (OUTEST=)';
run;

                               Step 2: Structural estimation via two-stage least squares                                


                       The SYSLIN Procedure

                  2SLS Estimation

  Instruments: CDD GASPRICE WINDINDEX CAPACITY P_LAG1 D_LAG1

  Model demandeq Dependent Variable: DEMAND

  Number of Observations                       60
  SSE                                  26254.9244
  MSE                                  468.837936
  R-Square                                 0.9628
  Adj R-Sq                                 0.9608

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1    597.362634   39.049548      15.30      0.0000
  PRICE          1      0.080161    0.617367       0.13      0.8972
  CDD            1      0.922369    0.046306      19.92      0.0000
  D_LAG1      

## Step 3 — Reduced form, multipliers, and dynamic simulation

Now we hand the structural estimates to **PROC SIMLIN**. Key pieces:

- `EST=struct TYPE=2sls` selects the 2SLS structural coefficients.
- `ENDOGENOUS` / `EXOGENOUS` declare the system layout; `LAGGED` maps each lag variable to its endogenous variable and lag order, which is what makes the simulation **dynamic** (predicted values feed the next period, not the actuals).
- `ESTPRINT` echoes the structural matrix; SIMLIN then inverts it to print the **reduced form** (each endogenous variable as a function of predetermined variables only).
- `INTERIM=3` reports multipliers 1–3 periods out; `TOTAL` reports the cumulative long-run multipliers once the lag dynamics fully dissipate.
- `OUTPUT OUT=` captures the simulated path; `OUTEST=redform` stores the reduced-form and multiplier matrices for downstream use.

In [4]:
title 'Step 3: Reduced form, multipliers & dynamic simulation (SIMLIN)';
proc simlin data=market_lags est=struct type=2sls
            estprint total interim=3 outest=redform;
   endogenous demand price renewgen;
   exogenous  cdd gasprice windindex capacity;
   lagged     p_lag1 price 1   d_lag1 demand 1;
   id month;
   output out=sim_out p=demand_hat price_hat renew_hat
                      r=demand_res price_res renew_res;
run;

                            Step 3: Reduced form, multipliers & dynamic simulation (SIMLIN)                             
                                        Structural coefficient dataset (OUTEST=)                                        


                       The SIMLIN Procedure

Model Information

  Endogenous Variables           3
  Estimation Type                2sls
  Exogenous Variables            4
  Number of Observations         60
  Total Lagged Coefficients      1

Structural Form Coefficients

  variable                     DEMAND          PRICE       RENEWGEN
  ---------------          ----------     ----------     ----------
  Intercept                597.362634       5.081764    -135.153194
  DEMAND                    -1.000000       0.021377       0.000000
  PRICE                      0.080161      -1.000000      -0.534652
  RENEWGEN                   0.000000      -0.026105      -1.000000
  P_LAG1                     0.000000       0.000000       0.000000
  D_LA

## Step 4 — Inspect the multiplier matrices

The `OUTEST=redform` dataset stores every coefficient matrix SIMLIN computed, tagged by `_TYPE_`. We filter to the **reduced form** (`REDUCED`) and the **impact multipliers** (`IMULT1`, the one-period response of each endogenous variable to a unit change in each exogenous driver). For example, the `GASPRICE` column under `PRICE` is the immediate pass-through of fuel costs into the wholesale price.

In [5]:
proc print data=redform;
   where _type_ = 'REDUCED' | _type_ = 'IMULT1';
   title2 'Reduced form and impact (period-1) multipliers';
run;

                            Step 3: Reduced form, multipliers & dynamic simulation (SIMLIN)                             
                                     Reduced form and impact (period-1) multipliers                                     

  Obs   _TYPE_  _DEPVAR_        INTERCEPT  P_LAG1         D_LAG1            CDD       GASPRICE       WINDINDEX       CAPACITY
-----  -------  --------  ---------------  ------  -------------  -------------  -------------  --------------  -------------
    1  REDUCED  DEMAND     599.1037424173       0   0.3073366845   0.9239743264    0.728531227   -0.9676851597  -0.0005486923
    2  REDUCED  PRICE       21.7200375676       0   0.0066628721   0.0200312005   9.0883033018  -12.0717079883  -0.0068448432
    3  REDUCED  RENEWGEN  -146.7658563387       0  -0.0035623182  -0.0107097225  -4.8590800191  461.6334996133   0.2617532597
    4  IMULT1   DEMAND                  .       .              .   0.9239743264    0.728531227   -0.9676851597  -0.0005486923
 

## Step 5 — Compare the simulated path to the actuals

The SIMLIN `OUT=` dataset holds the predicted (`*_hat`) and residual (`*_res`) series keyed by `month`. We merge those back against the observed market data so we can read actual-versus-simulated side by side for the first model year. A close track confirms the estimated structure reproduces the historical dynamics.

In [6]:
data compare;
   merge market_lags(keep=month demand price renewgen)
         sim_out(keep=month demand_hat price_hat renew_hat);
   by month;
run;

title 'Step 5: Actual vs simulated market path (first year)';
proc print data=compare(obs=12);
   var month demand demand_hat price price_hat renewgen renew_hat;
run;

                                  Step 5: Actual vs simulated market path (first year)                                  
                                     Reduced form and impact (period-1) multipliers                                     


  Obs  MONTH           DEMAND       DEMAND_HAT          PRICE      PRICE_HAT        RENEWGEN       RENEW_HAT
-----  -----  ---------------  ---------------  -------------  -------------  --------------  --------------
    1      1  1100.7754687553  1111.4350733141  47.8036620696  46.2638106501  394.0064595696    428.79328467
    2      2  1076.0038420773  1084.6437581038  43.5922680555  45.4166438806  414.5800012194  404.5150469737
    3      3  1053.4788793744  1046.8985239879  48.3541813638   44.457699758  481.6841417891  488.8705283172
    4      4  1013.9552809064  1025.2386853714  41.6934579666  43.5547638354  580.5437802996  554.6283141909
    5      5  1103.3020105045  1095.4428299165  46.4030389842  45.8690744829  534.5834858673  543.3770

## Interpretation

**Structure recovered.** The 2SLS stage returns coefficients with the expected signs: load rises with cooling demand and is persistent; wholesale price climbs with demand and gas cost but falls as renewables enter the stack; renewable output is driven chiefly by wind and installed capacity. Because the regressors are instrumented, these are consistent structural elasticities rather than the biased estimates single-equation OLS would give.

**Multipliers are the planning payoff.** The **impact (IMULT1)** multipliers say how load, price, and renewable output move *this month* when an exogenous driver shifts by one unit — e.g. the `GASPRICE → PRICE` entry is the immediate fuel-cost pass-through. The **interim** multipliers (periods 2–3) show the response decaying as the lag feedback propagates, and the **total** multipliers give the cumulative long-run effect once the dynamics settle. A planner reads the gap between impact and total multipliers as the difference between a price spike's first-month bite and its eventual full-system cost.

**Simulation validates the model.** The dynamic simulation feeds each period's *predicted* values into the next period's lags, so a tight actual-vs-simulated track (and small RMSE/MAE in SIMLIN's simulation statistics) means the structure is stable enough to use for scenario work — stress-testing a gas-price shock or a capacity expansion by editing the exogenous inputs and re-simulating, all without re-estimating the system.